# INSTRUCTOR SOLUTION: Model Versioning, Registry & Latency
## AIAT 125 — Unit 1: AI Model Deployment Lifecycle

**⚠️ INSTRUCTOR USE ONLY — Do not distribute to students**

| Task | Points |
|---|---|
| Task 1: Versioned checkpoint with simulated training | 25 |
| Task 2: Load and verify a checkpoint | 25 |
| Task 3: Model registry with promotion gate | 25 |
| Task 4: Latency benchmark P50/P95/P99 | 25 |

In [ ]:
import torch
import torch.nn as nn
import json
import time
import os
import numpy as np

SAVE_DIR = "/tmp/aiat125_unit1/"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Setup complete.")
print(f"PyTorch version : {torch.__version__}")
print(f"Save directory  : {SAVE_DIR}")

---
## Concept Demos (run before tasks)

In [ ]:
# Concept 1: versioned checkpoint demo
demo_model = nn.Linear(10, 2)
checkpoint_weights = demo_model.state_dict()
metadata = {"version": "v1.4.2", "developer": "AI_Team_Alpha",
            "training_loss": 0.045, "framework": "PyTorch 2.0"}
demo_save_path = os.path.join(SAVE_DIR, "model_v1_4_2.pth")
torch.save({'model_state': checkpoint_weights, 'metadata': metadata}, demo_save_path)
print(f"Demo checkpoint saved → {demo_save_path}")
print(f"Keys: {list(torch.load(demo_save_path, weights_only=False).keys())}")

In [ ]:
# Concept 2: auto-checkpoint demo
def auto_checkpoint(epoch, model_state, current_loss, best_loss=0.50):
    """Save checkpoint only when current_loss improves on best_loss."""
    if current_loss < best_loss:
        filename = os.path.join(SAVE_DIR, f"checkpoint_epoch_{epoch}.pth")
        torch.save({'model_state': model_state, 'epoch': epoch, 'loss': current_loss}, filename)
        print(f"  New best model found at epoch {epoch}! Saving {filename}...")
        return True
    return False

demo_losses = [0.80, 0.60, 0.45, 0.42, 0.48]
m = nn.Linear(10, 2)
print("Demo auto-checkpoint run:")
for ep, loss in enumerate(demo_losses, start=1):
    saved = auto_checkpoint(ep, m.state_dict(), loss)
    print(f"  Epoch {ep} | loss={loss:.3f} | {'SAVED' if saved else 'skipped'}")

In [ ]:
# Concept 3: model registry demo
model_registry = {
    "v1.0": "Deprecated - Too slow",
    "v1.1": "Production - Stable",
    "v1.2": "Staging - Undergoing latency tests"
}

def get_deployment_model(version_tag):
    status = model_registry.get(version_tag, "Version not found")
    print(f"Checking Registry: Version {version_tag} is currently {status}")
    return status

for v in ["v1.0", "v1.1", "v1.2", "v2.0"]:
    get_deployment_model(v)

In [ ]:
# Concept 4: latency testing demo
def run_latency_test(model, iterations=10):
    model.eval()
    latencies = []
    input_data = torch.randn(1, 10)
    for i in range(iterations):
        start_time = time.perf_counter()
        with torch.no_grad():
            _ = model(input_data)
        end_time = time.perf_counter()
        latencies.append((end_time - start_time) * 1000)
    avg_latency = sum(latencies) / len(latencies)
    p95_latency = sorted(latencies)[int(0.95 * len(latencies)) - 1]
    return avg_latency, p95_latency

m = nn.Linear(10, 2)
avg, p95 = run_latency_test(m, iterations=20)
print(f"Demo latency  →  avg={avg:.3f} ms  |  p95={p95:.3f} ms")

In [ ]:
# Concept 5: safety filter demo
def safety_test(model_output_text):
    restricted_content = ["malware", "hate", "violence"]
    return not any(word in model_output_text.lower() for word in restricted_content)

test_outputs = [
    "The weather today is sunny and warm.",
    "Here is how to write malware for beginners.",
    "I enjoy cooking and reading books.",
    "This content promotes violence against minorities."
]
for text in test_outputs:
    label = "SAFE" if safety_test(text) else "BLOCKED"
    print(f"  [{label}] {text[:60]}")

---
## Task 1 — Versioned Checkpoint with Simulated Training (25 points)

In [ ]:
TASK1_SAVE_PATH = os.path.join(SAVE_DIR, "task1_best_checkpoint.pth")

# SOLUTION 1a: Define a nn.Linear(10, 2) model
task1_model = nn.Linear(10, 2)

# Simulated epoch losses
epoch_losses = [0.72, 0.55, 0.38, 0.41, 0.36]

# SOLUTION 1b: Loop over epoch losses, track best
best_loss = float('inf')
best_epoch = None
best_model_state = None

for epoch, loss in enumerate(epoch_losses, start=1):
    if loss < best_loss:
        best_loss = loss
        best_epoch = epoch
        best_model_state = task1_model.state_dict()
        print(f"  Epoch {epoch}: new best loss = {loss:.2f}")
    else:
        print(f"  Epoch {epoch}: loss = {loss:.2f} (no improvement)")

# SOLUTION 1c: Build metadata dict
task1_metadata = {
    "version": f"v1.{best_epoch}.0",
    "developer": "AIAT125_Student",
    "training_loss": best_loss,
    "framework": f"PyTorch {torch.__version__}",
}

# SOLUTION 1d: Save the checkpoint
torch.save({'model_state': best_model_state, 'metadata': task1_metadata}, TASK1_SAVE_PATH)

print(f"Task 1 complete. Best epoch={best_epoch}, best_loss={best_loss}")
print(f"Checkpoint saved to: {TASK1_SAVE_PATH}")

In [ ]:
# Assertions: Task 1
assert task1_model is not None and isinstance(task1_model, nn.Linear)
assert best_epoch == 5, f"Expected best_epoch=5, got {best_epoch}"
assert abs(best_loss - 0.36) < 1e-6, f"Expected best_loss=0.36, got {best_loss}"
assert os.path.exists(TASK1_SAVE_PATH)
ckpt = torch.load(TASK1_SAVE_PATH, weights_only=False)
assert 'model_state' in ckpt and 'metadata' in ckpt
meta = ckpt['metadata']
for key in ['version', 'developer', 'training_loss', 'framework']:
    assert key in meta, f"metadata missing key: '{key}'"
assert meta['training_loss'] == best_loss
print("Task 1 PASSED — checkpoint saved with correct structure and best epoch.")

---
## Task 2 — Load and Verify a Checkpoint (25 points)

In [ ]:
REQUIRED_METADATA_KEYS = ['version', 'developer', 'training_loss', 'framework']

def load_and_verify_checkpoint(path):
    """
    Load a .pth checkpoint, verify its structure, and return (model_state, metadata).
    Raises FileNotFoundError or ValueError for invalid inputs.
    """
    # SOLUTION 2a: Raise FileNotFoundError if path does not exist
    if not os.path.exists(path):
        raise FileNotFoundError(f"Checkpoint not found: {path}")

    # SOLUTION 2b: Load the checkpoint
    checkpoint = torch.load(path, weights_only=False)

    # SOLUTION 2c: Verify 'model_state' and 'metadata' are present
    for key in ['model_state', 'metadata']:
        if key not in checkpoint:
            raise ValueError(f"Checkpoint missing required key: '{key}'")

    # SOLUTION 2d: Verify all required metadata keys
    for key in REQUIRED_METADATA_KEYS:
        if key not in checkpoint['metadata']:
            raise ValueError(f"metadata missing required key: '{key}'")

    # SOLUTION 2e: Return (model_state, metadata)
    return checkpoint['model_state'], checkpoint['metadata']


# SOLUTION 2f: Call function on TASK1_SAVE_PATH
task2_state, task2_meta = load_and_verify_checkpoint(TASK1_SAVE_PATH)
print("Loaded metadata:", task2_meta)

In [ ]:
# Assertions: Task 2
assert callable(load_and_verify_checkpoint)

try:
    load_and_verify_checkpoint("/tmp/this_file_does_not_exist.pth")
    assert False, "Should have raised FileNotFoundError"
except FileNotFoundError:
    pass

corrupt_path = os.path.join(SAVE_DIR, "corrupt.pth")
torch.save({'model_state': {}}, corrupt_path)
try:
    load_and_verify_checkpoint(corrupt_path)
    assert False, "Should have raised ValueError for missing 'metadata'"
except ValueError:
    pass

assert task2_state is not None and isinstance(task2_state, dict)
assert task2_meta  is not None and isinstance(task2_meta,  dict)
for key in REQUIRED_METADATA_KEYS:
    assert key in task2_meta, f"task2_meta missing: '{key}'"

print("Task 2 PASSED — load_and_verify_checkpoint works correctly.")

---
## Task 3 — Model Registry with Promotion Gate (25 points)

In [ ]:
# SOLUTION 3a: Define student_registry with three entries
student_registry = {
    "v2.0": "Deprecated - Performance below threshold",
    "v2.1": "Production - Stable and live",
    "v2.2": "Staging - Undergoing latency tests",
}


def promote_to_production(registry, version):
    """
    Promote a version from Staging to Production in the registry.
    Raises KeyError if version unknown, ValueError if not in Staging.
    """
    # SOLUTION 3b-i: Raise KeyError if version not in registry
    if version not in registry:
        raise KeyError(f"Version '{version}' not found in registry.")

    # SOLUTION 3b-ii: Raise ValueError if current status does not contain 'Staging'
    if "Staging" not in registry[version]:
        raise ValueError(
            f"Cannot promote '{version}': current status is '{registry[version]}'. "
            "Only Staging models can be promoted."
        )

    # SOLUTION 3b-iii: Update and return True
    registry[version] = "Production - Promoted"
    return True


# SOLUTION 3c: Promote v2.2
result = promote_to_production(student_registry, "v2.2")
print(f"Promotion result: {result}")
print("Updated registry:", student_registry)

In [ ]:
# Assertions: Task 3
assert isinstance(student_registry, dict) and len(student_registry) == 3
for v in ["v2.0", "v2.1", "v2.2"]:
    assert v in student_registry
assert "Deprecated" in student_registry["v2.0"]
assert "Production" in student_registry["v2.1"]
assert "Production" in student_registry["v2.2"]

try:
    promote_to_production(student_registry, "v9.9")
    assert False, "Should raise KeyError"
except KeyError:
    pass

try:
    promote_to_production(student_registry, "v2.1")
    assert False, "Should raise ValueError — v2.1 is Production"
except ValueError:
    pass

try:
    promote_to_production(student_registry, "v2.0")
    assert False, "Should raise ValueError — v2.0 is Deprecated"
except ValueError:
    pass

print("Task 3 PASSED — registry and promotion gate work correctly.")

---
## Task 4 — Latency Benchmark with P50 / P95 / P99 (25 points)

In [ ]:
NUM_ITERATIONS = 100

# SOLUTION 4a: Define benchmark_model in eval mode
benchmark_model = nn.Linear(10, 2)
benchmark_model.eval()

# SOLUTION 4b: Warm-up (5 passes, not recorded)
input_tensor = torch.randn(1, 10)
with torch.no_grad():
    for _ in range(5):
        _ = benchmark_model(input_tensor)

# SOLUTION 4c: Timed loop — 100 forward passes
latency_results = []
with torch.no_grad():
    for _ in range(NUM_ITERATIONS):
        t0 = time.perf_counter()
        _ = benchmark_model(input_tensor)
        t1 = time.perf_counter()
        latency_results.append((t1 - t0) * 1000.0)

# SOLUTION 4d: Sort and compute percentiles
sorted_latencies = sorted(latency_results)
p50 = sorted_latencies[int(0.50 * NUM_ITERATIONS) - 1]
p95 = sorted_latencies[int(0.95 * NUM_ITERATIONS) - 1]
p99 = sorted_latencies[int(0.99 * NUM_ITERATIONS) - 1]

print(f"Latency Benchmark ({NUM_ITERATIONS} iterations)")
print(f"  P50 (median) : {p50:.4f} ms")
print(f"  P95          : {p95:.4f} ms")
print(f"  P99          : {p99:.4f} ms")
print(f"  SLA (P95 < 100 ms): {'PASS' if p95 < 100 else 'FAIL'}")

In [ ]:
# Assertions: Task 4
assert benchmark_model is not None and isinstance(benchmark_model, nn.Linear)
assert not benchmark_model.training, "benchmark_model must be in eval mode"
assert isinstance(latency_results, list) and len(latency_results) == NUM_ITERATIONS
assert all(isinstance(x, float) and x > 0 for x in latency_results)
assert p50 is not None and p95 is not None and p99 is not None
assert p50 <= p95 <= p99
assert p95 < 100, f"p95 latency {p95:.4f} ms exceeds 100ms SLA"
print("Task 4 PASSED — latency benchmark complete, P95 < 100 ms SLA met.")

In [ ]:
# --- Final Deployment Gate ---
results = {}

try:
    assert isinstance(task1_model, nn.Linear)
    assert best_epoch == 5 and abs(best_loss - 0.36) < 1e-6
    ckpt1 = torch.load(TASK1_SAVE_PATH, weights_only=False)
    assert 'model_state' in ckpt1 and 'metadata' in ckpt1
    results['Task 1 - Versioned Checkpoint (25 pts)'] = 'PASS'
except Exception as e:
    results['Task 1 - Versioned Checkpoint (25 pts)'] = f'FAIL — {e}'

try:
    assert callable(load_and_verify_checkpoint)
    assert isinstance(task2_state, dict) and isinstance(task2_meta, dict)
    results['Task 2 - Load & Verify Checkpoint (25 pts)'] = 'PASS'
except Exception as e:
    results['Task 2 - Load & Verify Checkpoint (25 pts)'] = f'FAIL — {e}'

try:
    assert isinstance(student_registry, dict) and len(student_registry) == 3
    assert 'Deprecated' in student_registry.get('v2.0', '')
    assert 'Production' in student_registry.get('v2.2', '')
    results['Task 3 - Model Registry & Promotion Gate (25 pts)'] = 'PASS'
except Exception as e:
    results['Task 3 - Model Registry & Promotion Gate (25 pts)'] = f'FAIL — {e}'

try:
    assert isinstance(benchmark_model, nn.Linear) and not benchmark_model.training
    assert len(latency_results) == 100 and p95 < 100
    results['Task 4 - Latency Benchmark P50/P95/P99 (25 pts)'] = 'PASS'
except Exception as e:
    results['Task 4 - Latency Benchmark P50/P95/P99 (25 pts)'] = f'FAIL — {e}'

print("=" * 65)
print("  AIAT 125 — Unit 1: DEPLOYMENT GATE REPORT")
print("=" * 65)
total_pass = 0
for task, status in results.items():
    icon = "PASS" if status == 'PASS' else "FAIL"
    print(f"  [{icon}] {task}: {status}")
    if status == 'PASS':
        total_pass += 1
print("=" * 65)
print(f"  Score: {total_pass * 25} / 100  ({total_pass}/4 tasks passed)")
print("=" * 65)